# MLflow lab · Give María's model a memory

**MLOps · Master in Business Analytics and Data Science · MBDS-PT2026F · Forum 2, Optional Theme A**

In María's notebook, the model scored *"0.84 yesterday, 0.87 today, whatever"*, and nobody could say which settings, which data or which run produced which number. In this lab you train four versions of her churn model and let **MLflow** keep a record of every attempt, so you can compare them side by side.

**You do not need to write code.** You press ▶ on each grey cell, in order, and read the text between them. About 20 minutes.

| Step | What happens | Time |
|---|---|---|
| 1 | Install MLflow | 1–2 min |
| 2 | Load María's data | 1 min |
| 3 | Clean the data with fixed rules | seconds |
| 4 | Switch on MLflow's memory | seconds |
| 5 | Train three prepared models (A, B, C) | 30 s |
| 6 | Compare them | read and think |
| 7 | Your turn: train your own version | 5 min |
| 8 | *(Optional)* Open the MLflow screen we used in class | 2 min |
| 9 | What to post on the forum | |

**How to run a cell:** hover over the grey box and click ▶ on its left (or click inside it and press **Shift + Enter**). A green tick appears when it has finished. **If anything goes wrong:** menu **Runtime → Restart session**, then run the cells again from Step 1. Nothing you do here can break anything.

> The first time you press ▶, Google may warn that *"This notebook was not authored by Google"*. Click **Run anyway**. You may also be asked to sign in with a Google account.

## Step 1 · Install MLflow

Colab is a computer Google lends you for a few hours. It already has Python and the usual data libraries; we add MLflow. We ask for an **exact version** (3.15.2, the one from the class demo) so that everyone runs the same software: that is reproducibility in one line.

You will see some text scroll past (ignore any line saying *"you may need to restart"*). Wait for the green tick and the message *"MLflow is ready"*.

In [ ]:
%pip install -q mlflow==3.15.2
import mlflow
print("MLflow is ready, version", mlflow.__version__)

## Step 2 · Load María's data

The file is `churn_raw.csv` from the campus (the same one María's notebook loads): 1,212 customers of a subscription business, with their tenure, monthly charges, plan, contract, support tickets and whether they left (`churned = 1`).

When you run the cell, a **Choose files** button appears below it. Click it and pick `churn_raw.csv` from your computer. If you already dragged the file into the folder panel on the left, the cell will find it and skip the button.

In [ ]:
import os
import pandas as pd

if not os.path.exists("churn_raw.csv"):
    from google.colab import files
    print("Click 'Choose files' and select churn_raw.csv")
    files.upload()

raw = pd.read_csv("churn_raw.csv")
print(f"Loaded {len(raw):,} rows and {raw.shape[1]} columns.")
raw.head()

## Step 3 · Clean the data with fixed rules

María fixed her data by hand, differently each time. Here the same written rules run every time, and the cell prints a **receipt** of what they did. These are the validation and pre-processing boxes from Forum question 1:

1. **Duplicates:** a customer that appears twice is kept once.
2. **Impossible values:** a negative number of support tickets cannot happen, so those rows are set aside (not "fixed" with a guess).
3. **Error codes:** a monthly charge above 200 is the system's code for "unknown" (the 999s), so it becomes *empty*.
4. **Spelling:** `north` and `North` are the same region.
5. **Empty charges** are filled later, *inside* the model, using only the training customers (so the test customers never leak into the training).

At the end, the cell computes a **data fingerprint**: a short code that changes if even one value in the cleaned data changes. MLflow will store it with every run, so later you can prove which data a model was trained on.

*The code is hidden because the rules above are what matters. Click "Show code" if you are curious.*

In [ ]:
# @title Run the cleaning rules (click ▶) { display-mode: "form" }
import hashlib
import numpy as np

df = raw.copy()
receipt = {"rows received": len(df)}

before = len(df)
df = df.drop_duplicates(subset="CustomerID", keep="first")
receipt["duplicated customers removed"] = before - len(df)

before = len(df)
df = df[df["support_tickets"] >= 0].copy()
receipt["rows set aside (negative support tickets)"] = before - len(df)

receipt["charges above 200 marked as empty (error codes)"] = int((df["MonthlyCharges"] > 200).sum())
df.loc[df["MonthlyCharges"] > 200, "MonthlyCharges"] = np.nan

df["Region"] = df["Region"].str.strip().str.title()
df["is_monthly"] = (df["contract"] == "monthly").astype(int)
df["plan_level"] = df["plan"].map({"basic": 0, "standard": 1, "premium": 2})

FEATURES = ["tenure_months", "MonthlyCharges", "support_tickets", "is_monthly", "plan_level"]
TARGET = "churned"
df = df[["CustomerID"] + FEATURES + [TARGET]].reset_index(drop=True)

receipt["rows ready for training"] = len(df)
receipt["charges still empty (filled inside the model)"] = int(df["MonthlyCharges"].isna().sum())
receipt["share of customers who churned"] = f"{df[TARGET].mean():.0%}"

DATA_VERSION = "churn_raw-" + hashlib.sha256(df.to_csv(index=False).encode()).hexdigest()[:8]
receipt["data fingerprint (data version)"] = DATA_VERSION
CLEANING_RULES = "dedupe-id; drop-negative-tickets; charges>200=empty; region-title-case; median-fill-on-train"

print("CLEANING RECEIPT")
for k, v in receipt.items():
    print(f"  {k:<50} {v}")

## Step 4 · Switch on MLflow's memory

Below is the **whole training recipe**, written once and reused for every model. Look for the lines marked `# [MLflow]`: they are the only difference between this and María's notebook. They record, for every attempt (MLflow calls it a **run**):

- **parameters**: what was tried (algorithm, settings, which split of customers, which data version, which cleaning rules);
- **metrics**: how good it was;
- **tags**: who ran it and why;
- **the model file** itself.

You do not need to understand the code; just run the cell. It prints *"Recipe ready"*.

**The two scores we keep** (both between 0.5 and 1):

- **Score on studied customers** (*train AUC*): how well the model ranks customers it learned from.
- **Score on unseen customers** (*test AUC*): the same, on 30% of customers the model never saw. **This is the one that predicts real life.** 0.5 is a coin flip; 1.0 would be perfect.

We also keep **churners caught** (*recall*: of the customers who really left, the share the model flagged) and **alerts that were right** (*precision*: of the customers it flagged, the share who really left).

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
import mlflow.sklearn

mlflow.set_tracking_uri("sqlite:///mlflow.db")      # [MLflow] where the memory lives: one small file on this Colab machine
mlflow.set_experiment("churn-lab")                   # [MLflow] the folder all our runs go into


def train_and_log(run_name, algorithm="random_forest", max_depth=4, n_trees=200,
                  split_seed=42, author="professor", note=""):
    # 1. split the customers once: 70% to study, 30% kept unseen
    X_train, X_test, y_train, y_test = train_test_split(
        df[FEATURES], df[TARGET], test_size=0.3, random_state=split_seed, stratify=df[TARGET])

    # 2. choose the model (filling empty charges is part of the model, learned on the training customers only)
    if algorithm == "logistic_regression":
        model = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LogisticRegression(max_iter=1000))
        settings = {"algorithm": "logistic_regression"}
    else:
        depth = None if max_depth == 0 else max_depth
        model = make_pipeline(SimpleImputer(strategy="median"),
                              RandomForestClassifier(n_estimators=n_trees, max_depth=depth, random_state=1))
        settings = {"algorithm": "random_forest", "max_depth": "no limit" if depth is None else depth, "n_trees": n_trees}

    with mlflow.start_run(run_name=run_name):                                     # [MLflow] start recording this attempt
        model.fit(X_train, y_train)
        train_auc = roc_auc_score(y_train, model.predict_proba(X_train)[:, 1])
        test_auc = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])
        flagged = model.predict(X_test)
        recall = recall_score(y_test, flagged)
        precision = precision_score(y_test, flagged)

        mlflow.log_params({**settings, "split_seed": split_seed,                     # [MLflow] what was tried
                           "features": ", ".join(FEATURES), "data_version": DATA_VERSION,
                           "cleaning_rules": CLEANING_RULES, "train_rows": len(X_train)})
        mlflow.log_metrics({"test_auc": round(test_auc, 3), "train_auc": round(train_auc, 3),   # [MLflow] how good it was
                            "gap": round(train_auc - test_auc, 3),
                            "recall": round(recall, 3), "precision": round(precision, 3)})
        mlflow.set_tags({"author": author, "note": note})                            # [MLflow] who, and why
        mlflow.sklearn.log_model(model, name="model", input_example=X_train.head(3),  # [MLflow] the model file itself
                                 skops_trusted_types=["numpy.dtype", "sklearn.tree._tree.Tree"])
        #   (MLflow saves models in a safe format and asks us to vouch for two ordinary types; we built this model ourselves)

    print(f"Logged run '{run_name}':  unseen-customer score {test_auc:.3f}  |  studied-customer score {train_auc:.3f}")


print("Recipe ready. MLflow will store every run in mlflow.db, experiment 'churn-lab'.")

## Step 5 · Train three prepared models

Three versions of the churn model, from simple to very flexible:

- **A · simple-logistic**: a logistic regression, the simplest sensible model (a weighted score of the five columns).
- **B · forest-depth-4**: a random forest of 200 decision trees, each allowed only 4 questions deep.
- **C · forest-no-limit**: the same forest, but each tree may ask as many questions as it likes.

All three use the same customers, the same split and the same data version, so the comparison is fair. Takes about 30 seconds; you may see a few grey warnings, which you can ignore.

In [ ]:
import warnings, logging
warnings.filterwarnings("ignore")
logging.getLogger("mlflow").setLevel(logging.ERROR)

train_and_log("A · simple-logistic", algorithm="logistic_regression",
              note="simplest sensible baseline")
train_and_log("B · forest-depth-4", algorithm="random_forest", max_depth=4, n_trees=200,
              note="forest, kept shallow")
train_and_log("C · forest-no-limit", algorithm="random_forest", max_depth=0, n_trees=200,
              note="forest, no limit on depth")

## Step 6 · Compare the runs

This cell asks MLflow for **every run it remembers** and lays them side by side: first a table, then a chart. (We never typed the numbers in: MLflow kept them.) You can run this cell again at any time to refresh it.

In [ ]:
# @title Show the comparison (click ▶) { display-mode: "form" }
import matplotlib.pyplot as plt
from IPython.display import display

def show_comparison():
    runs = mlflow.search_runs(experiment_names=["churn-lab"], order_by=["start_time ASC"])
    def col(name):
        return runs[name] if name in runs.columns else ""
    table = pd.DataFrame({
        "run": runs["tags.mlflow.runName"],
        "author": col("tags.author"),
        "algorithm": col("params.algorithm"),
        "max depth": col("params.max_depth").fillna("–") if "params.max_depth" in runs else "–",
        "trees": col("params.n_trees").fillna("–") if "params.n_trees" in runs else "–",
        "split seed": col("params.split_seed"),
        "score on UNSEEN customers (test AUC)": runs["metrics.test_auc"],
        "score on studied customers (train AUC)": runs["metrics.train_auc"],
        "gap": runs["metrics.gap"],
        "churners caught (recall)": runs["metrics.recall"],
        "alerts that were right (precision)": runs["metrics.precision"],
        "data version": col("params.data_version"),
    })
    display(table.style.hide(axis="index")
                 .format(precision=3, subset=[c for c in table.columns if "(" in c or c == "gap"])
                 .background_gradient(subset=["score on UNSEEN customers (test AUC)"], cmap="Blues", vmin=0.5, vmax=0.8))

    # chart: studied vs unseen score per run
    names = list(table["run"])[::-1]
    test = list(table["score on UNSEEN customers (test AUC)"])[::-1]
    train = list(table["score on studied customers (train AUC)"])[::-1]
    y = np.arange(len(names))
    h = 0.36
    fig, ax = plt.subplots(figsize=(9, 1.1 + 0.75 * len(names)))
    ax.barh(y + h / 2 + 0.01, test, height=h, color="#2a78d6", label="UNSEEN customers (test): the one that matters")
    ax.barh(y - h / 2 - 0.01, train, height=h, color="#eb6834", label="studied customers (train)")
    for yi, a, b in zip(y, train, test):
        ax.text(b + 0.005, yi + h / 2 + 0.01, f"{b:.2f}", va="center", fontsize=9, color="#222", fontweight="bold")
        ax.text(a + 0.005, yi - h / 2 - 0.01, f"{a:.2f}", va="center", fontsize=9, color="#555")
    ax.axvline(0.5, color="#999", lw=1, ls="--")
    ax.text(0.502, len(names) - 0.45, "coin flip", fontsize=8, color="#777")
    ax.set_yticks(y, names)
    ax.set_xlim(0.45, 1.06)
    ax.set_xlabel("AUC score (0.5 = coin flip, 1.0 = perfect)")
    ax.set_title("Same data, different models: which one would you trust on new customers?", loc="left", fontsize=11)
    for s in ["top", "right"]:
        ax.spines[s].set_visible(False)
    ax.grid(axis="x", color="#eee")
    ax.set_axisbelow(True)
    ax.legend(loc="upper left", bbox_to_anchor=(0, -0.12 - 0.3 / len(names)), ncol=2, frameon=False, fontsize=9)
    plt.tight_layout()
    plt.show()

show_comparison()

### Look at the table and the chart before you go on

1. Which model has the **highest score on studied customers**? Is it also the best on **unseen** customers?
2. Look at the **gap** column. What do you think a big gap means? (Hint: think of a student who memorises last year's exam answers.)
3. Which model would you send to QA, and what would you say to someone who points at C's 1.00?

Keep your answers: they are your forum post.

## Step 7 · Your turn: train your own version

Change the settings in the form on the right of the next cell (drop-down, sliders, text boxes) and press ▶. Your run is added to MLflow's memory; the comparison table is shown again underneath with your run at the bottom.

**Two experiments worth trying** (one run each; you can run the cell as many times as you like, just change the run name):

- **"María's 0.84 vs 0.87":** keep everything as in model **B** (random forest, depth 4, 200 trees), but change only the **split seed** (for example to 7 or 123). The model is identical; only *which customers ended up in the unseen 30%* changes. What happens to the score?
- **Find the sweet spot:** try a forest with depth 2, 6 or 10. Where does the unseen score stop improving while the gap keeps growing?

Settings: `max_depth = 0` means *no limit*; `max_depth` and `n_trees` are ignored for logistic regression.

In [ ]:
# @title Your run { display-mode: "form" }
run_name = "D · my-run"  # @param {type:"string"}
your_initials = "XX"  # @param {type:"string"}
algorithm = "random_forest"  # @param ["random_forest", "logistic_regression"]
max_depth = 4  # @param {type:"slider", min:0, max:20, step:1}
n_trees = 200  # @param {type:"slider", min:10, max:500, step:10}
split_seed = 7  # @param {type:"integer"}
why = "same as B, different split seed"  # @param {type:"string"}

train_and_log(run_name, algorithm=algorithm, max_depth=max_depth, n_trees=n_trees,
              split_seed=split_seed, author=your_initials, note=why)
show_comparison()

## Step 8 · *(Optional)* Open the MLflow screen we used in class

Everything you need for the forum is already in the table above. If you want to see the same runs in the MLflow interface from the Session 3 demo, run the cell below and click the link it prints (it opens in a new browser tab). In the MLflow screen: **Experiments → churn-lab**, tick the runs and click **Compare**.

If the link shows an error or a blank page, do not worry: Colab sometimes blocks it. Skip this step.

In [ ]:
# @title Start the MLflow screen (optional) { display-mode: "form" }
import subprocess, time
from google.colab import output

# This Colab machine is private to you and is deleted when you close the tab,
# so we let the Colab browser link reach MLflow ("*" would never be used on a real company server).
subprocess.Popen(["mlflow", "ui", "--backend-store-uri", "sqlite:///mlflow.db", "--port", "5000",
                  "--workers", "1", "--allowed-hosts", "*", "--cors-allowed-origins", "*"],
                 stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(12)
output.serve_kernel_port_as_window(5000, path="/#/experiments")
print("Click the link above. In MLflow: Experiments → churn-lab → tick runs → Compare.")

## Step 9 · What to post in the Optional Theme A thread

Post **one screenshot** of your comparison table (or chart) that includes your own run, and **120–200 words** answering:

1. **Which run would you send to QA, and why?** Not "the highest number": say *which* number you trusted and why the others mislead.
2. **Your own run.** What did you change, and what happened? If you only changed the split seed, what does that tell you about María's *"0.84 yesterday, 0.87 today"*?
3. **What is still missing?** Look at what MLflow recorded for each run. Six months from now, your manager asks *"which model is live, and who said it was good enough?"* What does this record answer, and what would you still need that MLflow did not capture here?

Then reply to one classmate whose choice differs from yours.

*Declare AI use (or no use) as usual. Your Colab notebook disappears when you close it; nothing is saved on Google's side unless you choose File → Save a copy in Drive.*